# reads t1, inserts into t2, sends to Kafka

In [ ]:
import psycopg2
import uuid
from confluent_kafka import Producer

In [ ]:
#This cell is used to connect to the postgres database

connection=psycopg2.connect(
    host="localhost",
    database="pdf_pipeline",
    user="postgres",
    port="5432",
    password="root"
)
cursor=connection.cursor()

In [14]:
#This cell is used to define producer and connect to the kafka server

producer=Producer({'bootstrap.servers':"localhost:9092"})

In [15]:
# Running a loop over 'id' column in table t1 where 'is_active' is true and inserting the uuid into 'id' column of table t2 and sending that id to kafka topic.

cursor.execute("SELECT id FROM t1 WHERE is_active=True")
for rows in cursor.fetchall():
    id=rows[0]
    cursor.execute("INSERT INTO t2(id) VALUES (%s)",(id,))
    producer.produce("pdf_processed",str(id)) 
    #The , is used to convert id into a tuple as the produce function takes only one argument and if we do not put , then it will consider id as a string and will give error.
    producer.flush()
connection.commit()

In [16]:
cursor.close()
connection.close()